In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

### Linear regression summary (matrix form)

**Notation:**  
$A \in \mathbb{R}^{m\times n}$ — design matrix, $y \in \mathbb{R}^{m\times 1}$ — response vector  
$\hat{\beta}\in\mathbb{R}^{n\times 1}$ — parameter estimates, $\hat{y}=A\hat{\beta}$ — fitted values  
$m$ — number of observations, $n$ — number of parameters, $\alpha$ — significance level

| Quantity | Formula | NumPy / SciPy |
|---|---|---|
| Parameter estimates | $$\hat{\beta} = (A^{T}A)^{-1}A^{T}y$$ | `beta = np.linalg.lstsq(A, y, rcond=None)[0]` *(preferred)*<br><br>`beta = np.linalg.inv(A.T @ A) @ (A.T @ y)` |
| Residuals | $\varepsilon = y - A\hat{\beta}$ | `res = y - A @ beta` |
| Model standard deviation | $\hat{\sigma}=\sqrt{\dfrac{\varepsilon^{T}\varepsilon}{m-n}}$ | `sig_mod = np.sqrt((res.T @ res) / (m - n)).item()` |
| Fisher information matrix | $F=(A^{T}A)^{-1}$ | `F = np.linalg.inv(A.T @ A)` |
| $t$ critical value | $t_{1-\alpha/2,\;m-n}$ | `tcrit =  scipy.stats.t.ppf(1 - alpha/2, m - n)` |
| Confidence interval half-width for $\beta_i$ | $\Delta_i=t_{1-\alpha/2,\;m-n}\,\hat{\sigma}\sqrt{F_{ii}}$ | `Delta_i = (tcrit * sig_mod * np.sqrt(F[i, i])).item()` |
| Confidence interval for $\beta_i$ | $\hat{\beta}_i \pm \Delta_i$ | `ci_i = (beta[i,0] - Delta_i, beta[i,0] + Delta_i)` |
| Predicted value at $a_x$ | $\hat{y}=a_x\hat{\beta}$ | `y_hat = (axd @ beta).item()` |
| Mean response confidence interval half-width | $\Delta=t_{1-\alpha/2,\;m-n}\,\hat{\sigma}\sqrt{a_x(A^{T}A)^{-1}a_x^{T}}$ | `Delta = (tcrit * sig_mod * np.sqrt((axd @ F @ axd.T).item()))` |
| Mean response confidence interval | $\hat{y}\pm \Delta$ | `ci_mean = (y_hat - Delta, y_hat + Delta)` |
| Predictive confidence interval half-width | $\Delta=t_{1-\alpha/2,\;m-n}\,\hat{\sigma}\sqrt{1+a_x(A^{T}A)^{-1}a_x^{T}}$ | `Delta = (tcrit * sig_mod * np.sqrt((1 + (axd @ F @ axd.T).item())))` |
| Predictive confidence interval | $\hat{y}\pm \Delta$ | `ci_pred = (y_hat - Delta, y_hat + Delta)` |
| SSR | $SSR=\sum(\hat{y}_i-\bar{y})^2$ | `yhat = A @ beta`<br><br>`SSR = np.sum((yhat - y.mean())**2).item()` |
| SSE | $SSE=\sum(y_i-\hat{y}_i)^2=\varepsilon^{T}\varepsilon$ | `SSE = (res.T @ res).item()` |
| TSS | $TSS=\sum(y_i-\bar{y})^2 = SSR + SSE$ | `TSS = np.sum((y - y.mean())**2).item()` |
| F-statistic | $F=\dfrac{SSR/k}{SSE/(m-n)}$ | `Fstat = (SSR/k) / (SSE/(m-n))` |
| $R^2$ | $$R^2=1-\dfrac{SSE}{TSS}$$ | `R2 = (1 - SSE/TSS)` |
| Adjusted $R^2$ | $$R^2_{\mathrm{adj}} = 1 - \dfrac{m-1}{m-n}\dfrac{SSE}{TSS}$$ | `R2_adj = 1 - ((m-1)/(m-n))*(SSE/TSS)` |

*Note:* $k$ is the number of model terms (predictors) being tested in the numerator (depends on the hypothesis setup).


### Weighted linear regression summary (matrix form)

**Notation:**  
$A \in \mathbb{R}^{m\times n}$ — design matrix, $y \in \mathbb{R}^{m\times 1}$ — response vector  
$W=\mathrm{diag}(w_1,\dots,w_m)$ — weight matrix, $w_i>0$ — observation weights  
$\hat{\beta}\in\mathbb{R}^{n\times 1}$ — parameter estimates, $\hat{y}=A\hat{\beta}$ — fitted values  
$m$ — number of observations, $n$ — number of parameters, $\alpha$ — significance level  
$\bar{y}_w=\dfrac{\sum_i w_i y_i}{\sum_i w_i}$ — weighted mean of $y$

| Quantity | Formula | NumPy / SciPy |
|---|---|---|
| Weighted parameter estimates | $$\hat{\beta}=(A^{T}WA)^{-1}A^{T}Wy$$ | `Aw = A * np.sqrt(w)`<br><br>`yw = y * np.sqrt(w)`<br><br>`beta = np.linalg.lstsq(Aw, yw, rcond=None)[0]` *(preferred)* |
| (Equivalent normal-equation form) | $$\hat{\beta}=(A^{T}WA)^{-1}A^{T}Wy$$ | `AtWA = A.T @ (w * A)`<br><br>`AtWy = A.T @ (w * y)`<br><br>`beta = np.linalg.inv(AtWA) @ AtWy` |
| Predicted values | $\hat{y}=A\hat{\beta}$ | `yhat = A @ beta` |
| Weighted residuals | $$\varepsilon_w=W^{1/2}(y-A\hat{\beta})$$ | `res_w = np.sqrt(w) * (y - yhat)` |
| Fisher information matrix | $$F=(A^{T}WA)^{-1}$$ | `F = np.linalg.inv(A.T @ (w * A))` |
| $t$ critical value | $$t_{1-\alpha/2,\;m-n}$$ | `tcrit = scipy.stats.t.ppf(1 - alpha/2, m - n)` |
| Confidence interval half-width for $\beta_i$ | $$\Delta_i=t_{1-\alpha/2,\;m-n}\,\hat{\sigma}\sqrt{F_{ii}}$$ | `Delta_i = (tcrit * sig_mod * np.sqrt(F[i, i])).item()` |
| Confidence interval for $\beta_i$ | $$\hat{\beta}_i \pm \Delta_i$$ | `ci_i = (beta[i,0] - Delta_i, beta[i,0] + Delta_i)` |
| Predicted value at $a_x$ | $$\hat{y}(a_x)=a_x\hat{\beta}$$ | `y_hat = (axd @ beta).item()` |
| Mean response confidence interval half-width | $$\Delta=t_{1-\alpha/2,\;m-n}\,\hat{\sigma}\sqrt{a_x(A^{T}WA)^{-1}a_x^{T}}$$ | `Delta = (tcrit * sig_mod * np.sqrt((axd @ F @ axd.T).item())).item()` |
| Mean response confidence interval | $$\hat{y}\pm \Delta$$ | `ci_mean = (y_hat - Delta, y_hat + Delta)` |
| Predictive confidence interval half-width | $$\Delta=t_{1-\alpha/2,\;m-n}\,\hat{\sigma}\sqrt{\frac{1}{w_0}+a_x(A^{T}WA)^{-1}a_x^{T}}$$ | `Delta = (tcrit * sig_mod * np.sqrt((1/w0 + (axd @ F @ axd.T).item()))).item()` |
| Predictive cinfudence interval | $$\hat{y}\pm \Delta$$ | `ci_pred = (y_hat - Delta, y_hat + Delta)` |
| Weighted SSE | $$SSE=\varepsilon_w^{T}\varepsilon_w=\sum_i w_i\,(y_i-\hat{y}_i)^2$$ | `SSE = (res_w.T @ res_w).item()` |
| Model variance (weighted MSE) | $$\hat{\sigma}^2=\dfrac{SSE}{m-n}$$ | `sigma2 = (SSE / (m - n))` |
| Model standard deviation | $$\hat{\sigma}=\sqrt{\hat{\sigma}^2}$$ | `sig_mod = float(np.sqrt(sigma2))` |
| Weighted SSR | $$SSR=\sum_i w_i(\hat{y}_i-\bar{y}_w)^2$$ | `ybar_w = (w.T @ y).item() / (w.T @ np.ones_like(y)).item()`<br><br>`SSR = np.sum(w * (yhat - ybar_w)**2).item()` |
| Weighted TSS | $$TSS=\sum_i w_i(y_i-\bar{y}_w)^2=SSR+SSE$$ | `TSS = np.sum(w * (y - ybar_w)**2).item()` |
| F-statistic | $$F=\dfrac{SSR/k}{SSE/(m-n)}$$ | `Fstat = (SSR/k) / (SSE/(m-n))` |
| F-critical | $$F_{1-\alpha,\;k,\;m-n}$$ | `Fcrit = scipy.stats.f.ppf(1 - alpha, k, m - n)` |
| $R^2$ (weighted) | $$R^2=\dfrac{SSR}{TSS}=1-\dfrac{SSE}{TSS}$$ | `R2 = 1 - SSE/TSS` |
| Adjusted $R^2$ | $$R^2_{\mathrm{adj}} = 1 - \dfrac{m-1}{m-n}\dfrac{SSE}{TSS}$$ | `R2_adj = 1 - ((m-1)/(m-n))*(SSE/TSS)` |

*Note:* $k$ is the number of model terms (predictors) tested in the numerator (depends on the hypothesis setup).  
*Note:* For prediction intervals, $w_0$ is the weight associated with the new observation (same units/meaning as $w_i$).


### Nonlinear regression

In order to perform nonlinear regression, since nonlinear least-squares problems typically do not have a closed-form solution, in Python a numerical optimizer that could minimize an objective function is required. Once the function minimum is found, the optimal parameter estimates can be used to compute fitted values, residuals, and, with additional derivative information, approximate uncertainty measures such as standard errors and confidence intervals. In Python, the solver tools from `scipy.optimize` module can be used for the task.

It would be easier to define the method that would, given parameters, estimate the sum of residuals sqared

In [ ]:
def srs(beta, x, y):
    # beta: vector of parameter values
    # x: regressors for all experiments
    # y: measured values for all experiments
    yhat = model(x, beta)               # predicted values
    return np.sum((y - yhat)**2)        # sum of residuals squared

def model(x, beta):
    # example: yhat = B * x**b
    b_star, B = beta
    return B * x**b_star


#### Summary (matrix form)

**Notation:**  
$x \in \mathbb{R}^{m\times 1}$ — regressor values, $y \in \mathbb{R}^{m\times 1}$ — measured values  
$\hat{\beta}\in\mathbb{R}^{n\times 1}$ — estimated parameters, $\hat{y}=g(\hat{\beta},x)$ — fitted values  
$r = y-\hat{y}$ — residual vector, $m$ — number of observations, $n$ — number of parameters, $\alpha$ — significance level  
$J \in \mathbb{R}^{m\times n}$ — Jacobian of residuals, $J_{ij}=\dfrac{\partial r_i}{\partial \beta_j}$

| Quantity | Formula | NumPy / SciPy |
|---|---|---|
| Predicted values | $$\hat{y}=g(\hat{\beta},x)$$ | `yhat = model(x, beta)` |
| Residuals | $$r=y-\hat{y}$$ | `res = y - yhat` |
| Model standard deviation | $$\hat{\sigma}=\sqrt{\dfrac{\sum_{i=1}^{m} r_i^2}{m-n}}=\sqrt{\dfrac{r^{T}r}{m-n}}$$ | `sig_mod = np.sqrt((res.T @ res) / (m - n)).item()` |
| Fisher information matrix | $$F=(J^{T}J)^{-1}$$ | `F = np.linalg.inv(J.T @ J)` |
| $t$ critical value | $$t_{1-\alpha/2,\;m-n}$$ | `tcrit = scipy.stats.t.ppf(1 - alpha/2, m - n)` |
| confidence interval half-width for $\beta_i$ | $$\Delta_i=t_{1-\alpha/2,\;m-n}\,\hat{\sigma}\sqrt{F_{ii}}$$ | `Delta_i = (tcrit * sig_mod * np.sqrt(F[i, i])).item()` |
| Confidence interval for $\beta_i$ | $$\hat{\beta}_i \pm \Delta_i$$ | `ci_i = (beta[i,0] - Delta_i, beta[i,0] + Delta_i)` |
| Predicted value at $x_d$ | $$\hat{y}_d=g(\hat{\beta},x_d)$$ | `y_hat = model(xd, beta)` |
| Mean response confidence interval half-width | $$\Delta=t_{1-\alpha/2,\;m-n}\,\hat{\sigma}\sqrt{J_x\,F\,J_x^{T}}$$ | `Delta = (tcrit * sig_mod * np.sqrt((Jx @ F @ Jx.T).item())).item()` |
| Mean response CI | $$\hat{y}_d \pm \Delta$$ | `ci_mean = (y_hat - Delta, y_hat + Delta)` |
| Predictive confidence interval half-width | $$\Delta=t_{1-\alpha/2,\;m-n}\,\hat{\sigma}\sqrt{1+J_x\,F\,J_x^{T}}$$ | `Delta = (tcrit * sig_mod * np.sqrt((1 + (Jx @ F @ Jx.T).item()))).item()` |
| Predictive confidence interval | $$\hat{y}_d \pm \Delta$$ | `ci_pred = (y_hat - Delta, y_hat + Delta)` |
| SSR | $$SSR=\sum_{i=1}^{m}(\hat{y}_i-\bar{y})^2$$ | `SSR = np.sum((yhat - y.mean())**2).item()` |
| SSE | $$SSE=\sum_{i=1}^{m}(y_i-\hat{y}_i)^2=r^{T}r$$ | `SSE = (res.T @ res).item()` |
| TSS | $$TSS=\sum_{i=1}^{m}(y_i-\bar{y})^2 = SSR + SSE$$ | `TSS = np.sum((y - y.mean())**2).item()` |
| F-statistic | $$F=\dfrac{SSR/k}{SSE/(m-n)}$$ | `Fstat = (SSR/k) / (SSE/(m-n))` |
| F-critical | $$F_{1-\alpha,\;k,\;m-n}$$ | `Fcrit = scipy.stats.f.ppf(1 - alpha, k, m - n)` |
| $R^2$ | $$R^2=\frac{SSR}{TSS}=1-\frac{SSE}{TSS}$$ | `R2 = 1 - SSE/TSS` |
| Adjusted $R^2$ | $$R^2_{\mathrm{adj}} = 1 - \frac{m-1}{m-n}\frac{SSE}{TSS}$$ | `R2_adj = 1 - ((m-1)/(m-n))*(SSE/TSS)` |

*Note:* $k$ is the number of model terms (predictors) tested in the numerator (depends on the hypothesis setup).  
*Note:* $J_x$ denotes the Jacobian row vector evaluated at the prediction point $x_d$: $J_x=\left.\dfrac{\partial r}{\partial \beta}\right|_{x=x_d,\ \beta=\hat{\beta}}$.
